In [1]:
print(123)

123


In [2]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll?"

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
v1 = model.encode(q1)

In [5]:
v1.shape

(384,)

In [6]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [7]:
v1.dot(dv)

np.float32(0.39572883)

In [8]:
v1

array([-7.94801954e-03, -9.18932483e-02, -1.14074098e-02,  2.18466446e-02,
        1.11858072e-02, -1.30818337e-02, -7.39962384e-02, -9.87467393e-02,
       -1.05911404e-01, -3.03381961e-02, -2.92174350e-02,  2.33883206e-02,
        7.87485484e-03,  4.15800251e-02,  2.42720619e-02, -3.65723670e-02,
       -5.31095527e-02, -1.94868986e-02, -2.26979852e-02,  8.76781158e-03,
       -1.10785283e-01,  3.97906676e-02, -4.18480039e-02,  2.82960795e-02,
       -1.28302453e-02,  1.72567442e-02,  3.10428198e-02,  9.51102898e-02,
       -1.90717876e-02, -6.23235814e-02, -3.59101146e-02,  6.46180511e-02,
        3.06465030e-02,  2.39972137e-02,  2.06127614e-02,  9.87384189e-03,
        7.63835981e-02, -6.50510937e-02,  4.07932932e-03,  2.34527402e-02,
       -2.49407068e-02, -2.95020565e-02, -1.74879469e-02,  4.62139286e-02,
        2.48922799e-02,  1.08981758e-01, -5.67837432e-02, -6.95324540e-02,
        4.35404200e-03,  2.85132304e-02,  2.75795199e-02, -2.49843970e-02,
       -6.82142610e-03,  

In [9]:
import sys
sys.path.insert(0, '../01-agentic-rag')

from ingest import load_faq_data

documents = load_faq_data()

Generating embeddings

In [10]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [11]:
from tqdm.auto import tqdm

In [12]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [13]:
import numpy as np
X = np.array(vectors)

In [14]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [25]:
scores = X.dot(v_query)

Finding a best match


In [17]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.76294106))

In [18]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

Top 5 results

In [19]:
top5 = np.argsort(scores)[-5:]

In [23]:
top5 = top5[::-1]
top5

array([  7, 538, 907, 625,   2])

In [26]:
scores[top5]

array([0.56009996, 0.6536312 , 0.7192132 , 0.7579371 , 0.762941  ],
      dtype=float32)

In [28]:
top5 = np.argsort(-scores)[:5]
top5

array([  2, 625, 907, 538,   7])

In [29]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579371
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192132
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related 